In [1]:
import warnings
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import statsmodels.api as sm

from scipy import stats
from numpy.linalg import LinAlgError

warnings.filterwarnings('ignore')
%matplotlib inline
%load_ext autoreload
%autoreload 2

pd.options.display.float_format = "{:,.3f}".format
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

POSCLASS = 'COMBI'
NEGCLASS = 'PPCG'
INFILE_POSMUTS = '/home/grace/work/PPCG_DifferentialGenesetMutation/outputs/combi_ppcg_reactome/contrasts/COMBI/split_mutations/posmuts.tsv'
INFILE_NEGMUTS = '/home/grace/work/PPCG_DifferentialGenesetMutation/outputs/combi_ppcg_reactome/contrasts/COMBI/split_mutations/negmuts.tsv'

In [2]:
# mutations
pmuts = pd.read_csv(INFILE_POSMUTS, sep='\t', header=0)
nmuts = pd.read_csv(INFILE_NEGMUTS, sep='\t', header=0)
pmuts['cls'] = 'positive'
nmuts['cls'] = 'negative'
df = pd.concat([pmuts, nmuts], ignore_index=True)
df = df[df['vtype']!='CNApeak'].copy()
print(df.groupby('cls')['donor'].nunique())
df.head()

cls
negative    10
positive     2
Name: donor, dtype: int64


,donor,coords,ref,alt,gene,vclass,vtype,annotation,est_ccf,cls
0,PPCG0087,10:104592389,G,T,CYP17A1,SNV,SNV,missense_variant,1.000,positive
1,PPCG0087,10:108337033,G,A,SORCS1,SNV,SNV,missense_variant,0.810,positive
2,PPCG0087,10:32573709,T,C,EPC1,SNV,SNV,missense_variant,0.900,positive
3,PPCG0087,10:48390208,C,T,RBP3,SNV,SNV,missense_variant,0.880,positive
4,PPCG0087,10:82757959-10:87994013,.,.,GRID1,SV,INV,transcript_ablation,NaN,positive


In [3]:
df['vclass'].value_counts()

vclass
SV       1131
SNV       418
INDEL      44
Name: count, dtype: int64

### Summary ###

In [4]:
print('------------------')
print('--- Basic Info ---')
print('------------------')
print(f"Positive class: {POSCLASS}")
print(f"Negative class: {NEGCLASS}")
print()
print(f"--- Donors ---")
print(f"Total:    {df['donor'].nunique()}")
print(f"Positive: {df[df['cls']=='positive']['donor'].nunique()}")
print(f"Negative: {df[df['cls']=='negative']['donor'].nunique()}")
print()
print(f"--- Genes ---")
print(f"Total:    {df['gene'].nunique()}")
print(f"Positive: {df[df['cls']=='positive']['gene'].nunique()}")
print(f"Negative: {df[df['cls']=='negative']['gene'].nunique()}")
print()
print(f"--- Variants ---")
print(f"Total:    {df['coords'].nunique()}")
print(f"Positive: {df[df['cls']=='positive']['coords'].nunique()}")
print(f"Negative: {df[df['cls']=='negative']['coords'].nunique()}")


------------------
--- Basic Info ---
------------------
Positive class: COMBI
Negative class: PPCG

--- Donors ---
Total:    12
Positive: 2
Negative: 10

--- Genes ---
Total:    1085
Positive: 261
Negative: 857

--- Variants ---
Total:    1253
Positive: 293
Negative: 960


In [5]:
print()
print('-----------------------------')
print('--- Variant Class Summary ---')
print('-----------------------------')

for clsmem in ['positive', 'negative']:
    print()
    print(f'[{clsmem.capitalize()}]')
    dfslice = df[df['cls']==clsmem]
    ndonors = dfslice['donor'].nunique()
    counts = pd.DataFrame(index=sorted(list(dfslice['vclass'].unique())))
    counts['variants'] = dfslice['vclass'].value_counts()
    counts['genes'] = dfslice.groupby('vclass')['gene'].nunique()
    counts['donors'] = dfslice.groupby('vclass')['donor'].nunique()
    counts['donors prop.'] = counts['donors'] / ndonors
    print(counts)


-----------------------------
--- Variant Class Summary ---
-----------------------------

[Positive]
       variants  genes  donors  donors prop.
INDEL         9      8       1         0.500
SNV         113    103       2         1.000
SV          258    155       2         1.000

[Negative]
       variants  genes  donors  donors prop.
INDEL        35     34      10         1.000
SNV         305    293      10         1.000
SV          873    542      10         1.000


In [6]:
print()
print('----------------------------------')
print('--- Variant Annotation Summary ---')
print('----------------------------------')
for clsmem in ['positive', 'negative']:
    print()
    print(f'[{clsmem.capitalize()}]')
    dfslice = df[df['cls']==clsmem]
    ndonors = dfslice['donor'].nunique()
    counts = pd.DataFrame(index=sorted(list(dfslice['annotation'].unique())))
    counts['variants'] = dfslice['annotation'].value_counts()
    counts['genes'] = dfslice.groupby('annotation')['gene'].nunique()
    counts['donors'] = dfslice.groupby('annotation')['donor'].nunique()
    counts['donors prop.'] = counts['donors'] / ndonors
    counts = counts.sort_values('variants', ascending=False)
    print(counts)


----------------------------------
--- Variant Annotation Summary ---
----------------------------------

[Positive]
                                variants  genes  donors  donors prop.
transcript_ablation                  105     70       2         1.000
missense_variant                      71     66       2         1.000
bidirectional_gene_fusion             68     52       2         1.000
gene_fusion&frameshift_variant        49     40       2         1.000
gene_fusion                           36     32       2         1.000
upstream_gene_variant                 17     17       2         1.000
downstream_gene_variant               13     11       2         1.000
stop_gained                           10     10       2         1.000
frameshift_variant                     5      5       1         0.500
inframe_variant                        3      3       1         0.500
unknown                                3      3       1         0.500

[Negative]
                              

In [7]:
print()
print('-----------------------------------')
print('--- Gene Summary (top 20 genes) ---')
print('-----------------------------------')
for clsmem in ['positive', 'negative']:
    print()
    print(f'[{clsmem.capitalize()}]')
    dfslice = df[df['cls']==clsmem]
    ndonors = dfslice['donor'].nunique()
    counts = pd.DataFrame(index=sorted(list(dfslice['gene'].unique())))
    counts['sv'] = dfslice[dfslice['vclass']=='SV']['gene'].value_counts()
    counts['snvs'] = dfslice[dfslice['vclass']=='SNV']['gene'].value_counts()
    counts['indels'] = dfslice[dfslice['vclass']=='INDEL']['gene'].value_counts()
    counts['cna'] = dfslice[dfslice['vclass']=='CNA']['gene'].value_counts()
    counts['total'] = dfslice['gene'].value_counts()
    counts['donors'] = dfslice.groupby('gene')['donor'].nunique()
    counts = counts.fillna(0).astype(int)
    counts['donors prop.'] = counts['donors'] / ndonors
    counts['donors prop.'] = counts['donors prop.'].apply(lambda x: f"{x*100:.1f} %")
    counts = counts.sort_values(by=['donors', 'total'], ascending=[False, False])
    print(counts.head(20))


-----------------------------------
--- Gene Summary (top 20 genes) ---
-----------------------------------

[Positive]
                 sv  snvs  indels  cna  total  donors donors prop.
FOXP1             4     0       0    0      4       2      100.0 %
ZCWPW2            4     0       0    0      4       2      100.0 %
ANK2              2     1       0    0      3       2      100.0 %
TP53              2     0       1    0      3       2      100.0 %
ERG               2     0       0    0      2       2      100.0 %
SORCS1            1     1       0    0      2       2      100.0 %
TMPRSS2           2     0       0    0      2       2      100.0 %
GRID1            19     0       0    0     19       1       50.0 %
EDEM2             4     4       0    0      8       1       50.0 %
MMP24-AS1-EDEM2   4     4       0    0      8       1       50.0 %
ACACA             6     0       0    0      6       1       50.0 %
APPL2             4     0       0    0      4       1       50.0 %
TUBGCP3 

### Differential Mutation ###

In [ ]:
def gen_matrix(table: pd.DataFrame) -> pd.DataFrame:
    # init dataframe
    all_donors = sorted(list(table['donor'].unique()))
    df = pd.DataFrame(index=all_donors)

    # annotate membership & TMB
    df['cls'] = table.drop_duplicates('donor').set_index('donor')['cls']
    df['TMB'] = table.groupby('donor')['gene'].nunique() #? coords or genes....hmmm
    df = df.reset_index()
    df = df.rename(columns={'index': 'donor'})
    df = df[['cls', 'donor', 'TMB']].copy()
    df = df.sort_values('cls', ascending=False)

    # for each geneset, add column annotating whether the donor has a mutation or not. 
    i = 0
    for gene, donors in table.groupby('gene')['donor'].agg(set).to_dict().items():
        df[gene] = df['donor'].apply(lambda x: 1 if x in donors else 0)
        i += 1
        if i % 50 == 0:
            df = df.copy()
    return df

def evaluate(df: pd.DataFrame) -> pd.DataFrame:
    all_genes = df.columns.to_list()[3:]
    data = []
    for gene in all_genes:
        counts = df.groupby('cls')[gene].sum()
        fish_pval = _calc_fisher(df, gene)
        logit_pval, converged = _calc_logit(df, gene)
        data.append((gene, counts['positive'], counts['negative'], fish_pval, logit_pval, converged))

    rframe = pd.DataFrame.from_records(data, columns=['geneset', 'posclass donors', 'negclass donors', 'fisher_pval', 'logit_pval', 'logit_converged'])
    rframe = rframe.sort_values('fisher_pval')
    return rframe

def _calc_fisher(df: pd.DataFrame, gene: str) -> float:
    a = ((df[gene]==1) & (df['cls']=='positive')).sum()
    b = ((df[gene]==1) & (df['cls']=='negative')).sum()
    c = ((df[gene]==0) & (df['cls']=='positive')).sum()
    d = ((df[gene]==0) & (df['cls']=='negative')).sum()
    res = stats.fisher_exact([[a, b], [c, d]], alternative='two-sided')
    return float(res.pvalue)  # type: ignore

def _calc_logit(df: pd.DataFrame, gene: str):
    y = df[gene]
    X = pd.DataFrame({
        "cls": (df["cls"] == "positive").astype(int),
        "log_TMB": np.log1p(df["TMB"]) # log(1 + TMB)
    })
    X = sm.add_constant(X)
    try:
        model = sm.Logit(y, X).fit(disp=0)
        return float(model.pvalues['cls']), model.mle_retvals['converged']
    except LinAlgError:
        return 1.0, False


In [10]:
mat = gen_matrix(df)
print(mat.iloc[:, :5])
print()
print(mat.iloc[:, -5:])
res = evaluate(mat)

         cls     donor  TMB  AASS  ABCB1
10  positive  PPCG0087  149     0      0
11  positive  PPCG0088  119     0      0
0   negative  PPCG0001   30     1      0
1   negative  PPCG0003   49     0      0
2   negative  PPCG0004   73     0      0
3   negative  PPCG0006  144     0      1
4   negative  PPCG0008   96     0      0
5   negative  PPCG0010   85     0      0
6   negative  PPCG0017   89     0      0
7   negative  PPCG0019  212     0      0
8   negative  PPCG0022   78     0      0
9   negative  PPCG0052   62     0      0

    ZNF831  ZPBP  ZPLD1  ZRANB3  ZSWIM2
10       0     0      0       1       0
11       0     0      0       0       0
0        1     0      0       0       0
1        0     0      0       0       1
2        0     0      0       0       0
3        0     1      0       0       0
4        0     0      0       0       0
5        0     0      1       0       0
6        0     0      0       0       0
7        0     0      0       0       0
8        0     0      0   

In [13]:
res.head(20)

,geneset,posclass donors,negclass donors,fisher_pval,logit_pval,logit_converged
989,TP53,2,0,0.015,1.000,False
907,SORCS1,2,0,0.015,1.000,False
1059,ZCWPW2,2,0,0.015,1.000,False
53,ANK2,2,1,0.045,1.000,False
334,FOXP1,2,1,0.045,1.000,False
285,ERG,2,3,0.152,0.913,False
20,ADAM32,1,0,0.167,0.991,False
29,ADCY6-DT,1,0,0.167,0.991,False
898,SNRNP35,1,0,0.167,0.991,False
6,ABI3BP,1,0,0.167,0.991,False


### Background Enrichment ###